In [1]:
from astropy import table

import matplotlib.pyplot as plt
import numpy as np

from ugdatalab import (
    GaiaData,
    sanitize_vari_rrlyrae_table,
    attach_flux_mean_magnitudes,
    cross_validate_harmonics,
    fourier_fit,
    fourier_mean_magnitude,
    phase_fold,
    plot_rrlyrae_shape_comparison,
)

Some IP addresses of users launching heavy query showers have temporarily been disabled. Please contact the Gaia helpdesk (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk) for advice. Workaround solutions for the Gaia Archive issues following the infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive


In [2]:
rrc_query = """
SELECT TOP 3 *
FROM gaiadr3.vari_rrlyrae
WHERE p1_o IS NOT NULL
  AND int_average_g  < 15
  AND num_clean_epochs_g > 80
ORDER BY num_clean_epochs_g DESC
"""


In [3]:
rrc = GaiaData(rrc_query, include_lightcurve=True)


INFO: Query finished. [astroquery.utils.tap.core]


In [4]:
rrab_query = """
SELECT TOP 3 *
FROM gaiadr3.vari_rrlyrae
WHERE pf IS NOT NULL
  AND p1_o IS NULL
  AND int_average_g  < 15
  AND num_clean_epochs_g > 80
ORDER BY num_clean_epochs_g DESC
"""


In [5]:
rrab = GaiaData(rrab_query, include_lightcurve=True)


INFO: Query finished. [astroquery.utils.tap.core]


In [6]:
rrc_data = sanitize_vari_rrlyrae_table(rrc.data)
rrab_data = sanitize_vari_rrlyrae_table(rrab.data)
rrc_lightcurve_data = attach_flux_mean_magnitudes(rrc.lightcurve_data)
rrab_lightcurve_data = attach_flux_mean_magnitudes(rrab.lightcurve_data)

In [7]:
def prepare_class_summary_and_panels(sample_data, lightcurve_data, rr_class):
    period_column = "p1_o" if rr_class == "RRc" else "pf"
    lightcurve_source_ids = lightcurve_data["source_id"]

    summary_rows = []
    panels = []

    for row in sample_data:
        source_id = int(row["source_id"])
        period = float(row[period_column])

        star = lightcurve_data[lightcurve_source_ids == source_id]
        _, _, _, best_K, _, _ = cross_validate_harmonics(star, period)
        fit = fourier_fit(star, period, int(best_K))
        mean_fourier_g = fourier_mean_magnitude(fit)

        phase = phase_fold(star["g_transit_time"], period)
        mag_centered = np.asarray(star["g_transit_mag"], dtype=float) - mean_fourier_g
        order = np.argsort(phase)
        phase = phase[order]
        mag_centered = mag_centered[order]

        phase_grid = np.linspace(0.0, 1.0, 1000, endpoint=False)
        model_centered = fit.predict(phase_grid * period) - mean_fourier_g

        summary_rows.append(
            {
                "rr_class": rr_class,
                "source_id": source_id,
                "int_average_g": float(row["int_average_g"]),
                "num_clean_epochs_g": int(row["num_clean_epochs_g"]),
                "period": period,
                "best_K": int(best_K),
                "n_epochs": len(star),
            }
        )
        panels.append(
            {
                "source_id": source_id,
                "rr_class": rr_class,
                "phase_data": phase,
                "mag_centered": mag_centered,
                "phase_grid": phase_grid,
                "model_centered": model_centered,
                "best_K": int(best_K),
                "period": period,
                "n_epochs": len(star),
            }
        )

    return table.Table(rows=summary_rows), panels

In [8]:
rrc_summary, rrc_panels = prepare_class_summary_and_panels(rrc_data, rrc_lightcurve_data, "RRc")

rrc_summary

TypeError: cross_validate_harmonics() takes 1 positional argument but 2 were given

In [ ]:
rrab_summary, rrab_panels = prepare_class_summary_and_panels(rrab_data, rrab_lightcurve_data, "RRab")

rrab_summary

In [ ]:
summary = table.vstack([rrc_summary, rrab_summary])

summary["rr_class", "source_id", "int_average_g", "num_clean_epochs_g", "period", "best_K", "n_epochs"]

In [ ]:
axes = plot_rrlyrae_shape_comparison(rrc_panels + rrab_panels)
plt.show()

## Analysis and Discussion

The six phased light curves recover the standard class-dependent RR Lyrae morphology very clearly. The **RRab** stars are the more asymmetric objects: they show the familiar steep rise to maximum light, slower decline, and larger peak-to-peak amplitudes. The **RRc** stars are smoother and closer to sinusoidal, with smaller amplitudes and more symmetric light-curve shapes. That is exactly the expected phenomenology for the two main RR Lyrae subclasses: RRab stars pulsate in the **radial fundamental mode**, whereas RRc stars pulsate in the **radial first overtone**, and this difference in pulsation mode is reflected directly in the shape of the light curve (Clementini et al. 2023).

A single period plus a finite Fourier series is therefore a good **first-order** description of all six stars. The fitted curves capture the dominant phase dependence and correctly reproduce the broad RRab-versus-RRc contrast. In that limited sense, the six stars are all "well described" by a single-period model. However, the more careful question is whether they are described by a **strictly repeating** single period with only measurement noise about the model, or whether there is evidence for additional astrophysical scatter. The phased plots should be read with that distinction in mind.

For several of the stars, the points do not collapse onto an infinitely thin curve after phase folding. Instead, there is visible width around the Fourier template at fixed phase, especially near maxima, minima, or along the steepest parts of the variation. Some of that width is expected from Gaia photometric uncertainty and from the fact that the Fourier model is only a finite-order approximation. But if the scatter is systematically larger than the nominal point-to-point noise, or if groups of points appear displaced in phase or amplitude relative to the mean template, then the light curve is not perfectly described by a single coherent oscillator. In a folded plot, slow cycle-to-cycle changes show up as **intrinsic broadening** of the locus rather than as a second clean track.

That is exactly the kind of behaviour associated with departures from simple periodicity in RR Lyrae stars. Netzel et al. (2018) describe the **Blazhko effect** in RRc stars as a quasi-periodic modulation of pulsation **amplitude and/or phase**, and show that in the frequency domain it produces close side peaks, triplets, or more complicated multiplets around the main pulsation frequency and its harmonics. In the time domain, the practical consequence is that a light curve folded on a single period no longer looks perfectly single-valued: measurements from different cycles can land at slightly different magnitudes at the same nominal phase because the pulsation envelope is itself being modulated (Netzel et al. 2018). Their OGLE study found this effect in a non-negligible fraction of first-overtone RR Lyrae stars, detecting Blazhko modulation in **607 out of 10,826 RRc stars (5.6%)**, with modulation periods ranging from a few days to nearly 3000 days and with some stars showing **multiperiodic** modulation (Netzel et al. 2018).

That result matters directly for the present notebook. If one or more of the RRc panels shows excess phase broadening or systematic residual structure relative to the Fourier curve, it is reasonable to interpret that as possible evidence that the star is not a perfectly single-period oscillator. The same logic applies to RRab stars as well: Blazhko-type amplitude and phase modulation is long known among RR Lyrae stars more generally, and it can make a folded light curve appear noisier or broader than expected from measurement error alone (Buchler & Kolláth 2011). In other words, what looks like "scatter" in the phased plot need not be merely observational noise; it can be the projection of slow intrinsic modulation onto a single-period fold.

So the most defensible conclusion is that the six stars are **approximately**, but not always **perfectly**, described by a single period. The Fourier models capture the mean pulsation shape and are good summary descriptions, but any visible phase-dependent thickness of the point cloud is evidence that a strictly periodic one-period model is incomplete. The most plausible astrophysical explanation for that incompleteness is modulation of amplitude and/or phase, including possible Blazhko-like behaviour, rather than failure of the RRab/RRc classification itself. To establish that securely one would normally inspect the residual time series or the Fourier spectrum for side peaks around the main mode and its harmonics, but even the phased plots already provide a useful qualitative warning that some RR Lyrae light curves contain real deviations from simple periodic repetition (Netzel et al. 2018; Buchler & Kolláth 2011).

References: Clementini et al. 2023, *A\&A*, **674**, A18; Netzel et al. 2018, *MNRAS*, **480**, 1229; Buchler & Kolláth 2011, *ApJ*, **731**, 24.